In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
# Версии из отправленного архива кладутся поверх: именно на них обучена структурная модель.
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from src.features import extract_model_features
from src.model import BoostedPairModel
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata, spearmanr

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))
log(f"пар {len(pairs):,}, доля+ {y.mean():.3f}")

names = list(feature_names(False))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
for c in known:
    rows = np.flatnonzero(cat == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), known, with_neighbours=False)
    del sub; gc.collect()
log("парные признаки готовы")

t = time.perf_counter()
legacy = extract_model_features(pairs[["id1", "id2"]], items)
primary = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
aux = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
structural = (0.8 * primary + 0.2 * aux).astype(np.float32)
del legacy; gc.collect()
log(f"структурная модель за {time.perf_counter()-t:.0f}с")

M = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
MX = np.array([[r[n] for n in MEASURE_FEATURES] for r in
               (compare_measures(M[x], M[z]) for x, z in zip(pairs.id1, pairs.id2))], dtype=np.float32)
import re
IDENTIFIER_KEYS = (
    "sku", "код товара", "артикул", "арт", "партномер", "part number",
    "номер детали", "oem", "оем", "артикул производителя", "партномер производителя",
)
MODEL_KEYS = ("модель", "model", "серия", "линейка", "коллекция")
BRAND_KEYS = ("бренд", "brand", "производитель", "producer", "марка")
TYPE_KEYS = ("тип", "вид", "тип изделия", "вид изделия", "тип продукта", "вид товара", "назначение")
COLOR_KEYS = ("цвет", "color", "оттенок")

# More specific keys must appear before generic parent concepts.
SEMANTIC_KEY_RULES: tuple[tuple[str, tuple[str, ...]], ...] = (
    ("package_quantity", ("количество предметов в упаковке", "единиц в одном товаре", "количество в упаковке", "количество упаковок", "штук в упаковке", "количество ручек", "количество линз")),
    ("pet_size", ("размер животного", "размер птицы", "размер собаки", "размер кошки")),
    ("manufacturer_size", ("размер производителя", "eur размер", "размер обуви производителя")),
    ("shoe_size_ru", ("российский размер", "ru размер")),
    ("package_dimension", ("размер упаковки", "габарит упаковки", "длина упаковки", "ширина упаковки", "высота упаковки", "размеры упаковки")),
    ("product_dimension", ("габарит", "размеры", "размер товара", "длина", "ширина", "высота", "диаметр", "размер моста", "ширина линзы", "высота линзы", "длина заушника", "общая ширина")),
    ("weight", ("вес", "масса")),
    ("volume", ("объем", "вместимость")),
    ("power", ("мощность", "power")),
    ("voltage", ("напряжение", "voltage")),
    ("optical_power", ("оптическая сила", "диоптр", "sphere", "сфер")),
    ("cylinder", ("цилиндр", "cyl")),
    ("axis", ("ось", "axis", "ах")),
    ("radius", ("радиус кривизны", "радиус")),
    ("quantity", ("количество", "шт", "pcs", "units")),
    ("format", ("формат",)),
    ("density", ("плотность",)),
    ("type", TYPE_KEYS),
    ("brand", BRAND_KEYS),
    ("color", COLOR_KEYS),
    ("identifier", IDENTIFIER_KEYS),
    ("model", MODEL_KEYS),
    ("composition", ("состав", "ингредиенты", "материал", "материал изделия", "материал корпуса", "материал линз", "материал оправы")),
)

CATEGORY_CRITICAL_FIELDS = {
    "аптека": {"optical_power", "cylinder", "axis", "radius", "product_dimension", "package_quantity", "identifier", "model"},
    "автотовары": {"identifier", "model", "brand", "type", "product_dimension"},
    "обувь": {"model", "manufacturer_size", "shoe_size_ru", "color", "type", "brand"},
    "товары для животных": {"brand", "model", "pet_size", "package_quantity", "weight", "volume", "composition", "type"},
    "канцелярские товары": {"format", "product_dimension", "package_quantity", "density", "type", "brand"},
    "красота и гигиена": {"brand", "model", "volume", "weight", "composition", "type", "color"},
    "бытовая техника": {"brand", "model", "product_dimension", "volume", "weight", "power", "type", "identifier"},
}

FIELD_WEIGHT = {
    "identifier": 5.0, "model": 4.0, "type": 3.0, "package_quantity": 3.0,
    "weight": 3.0, "volume": 3.0, "shoe_size_ru": 3.0, "manufacturer_size": 3.0,
    "optical_power": 5.0, "cylinder": 5.0, "axis": 5.0, "radius": 4.0,
    "pet_size": 2.5, "product_dimension": 2.5, "brand": 2.0, "composition": 2.0,
    "color": 1.0, "format": 1.5, "density": 1.5, "power": 2.0, "voltage": 2.0,
    "quantity": 2.5,
}

# Канонические поля: ключи атрибутов приводятся к смыслу, у каждой категории свои
# решающие поля, у полей веса. Таблицы взяты из ветки сокомандника — это предметное
# знание, которого у нас не было: наш разбор сваливает артикул, партномер и oem в один
# слот и считает все расхождения одинаковыми.
SEM_RE = tuple((name, tuple(k.lower() for k in keys)) for name, keys in SEMANTIC_KEY_RULES)
NUM_RE = re.compile(r"-?\d+(?:[.,]\d+)?")
TOK_RE = re.compile(r"[a-zа-яё0-9]+")

def semantic_of(key):
    low = key.lower()
    for name, keys in SEM_RE:
        if any(k in low for k in keys):
            return name
    return None

def canon(raw):
    try:
        d = json.loads(raw) if raw else {}
    except Exception:
        return {}, {}
    fields, nums = {}, {}
    for k, v in d.items():
        s = semantic_of(str(k))
        if s is None:
            continue
        text = str(v).lower()
        fields.setdefault(s, set()).update(TOK_RE.findall(text))
        found = NUM_RE.findall(text.replace(",", "."))
        if found:
            nums.setdefault(s, set()).add(float(found[0]))
    return {k: frozenset(v) for k, v in fields.items()}, {k: frozenset(v) for k, v in nums.items()}

CANON = {int(i): canon(a) for i, a in zip(items.id, items.attributes)}
log("канонические поля построены")
crit_of = {}
for c in known:
    low = c.lower()
    picked = None
    for cat_name, fields in CATEGORY_CRITICAL_FIELDS.items():
        if cat_name in low or low in cat_name:
            picked = set(fields); break
    crit_of[c] = picked or {"identifier", "model", "type", "package_quantity",
                            "weight", "volume", "product_dimension"}

def canon_features(x, z, category):
    lf, ln = CANON.get(int(x), ({}, {}))
    rf, rn = CANON.get(int(z), ({}, {}))
    crit = crit_of.get(category, set())
    shared = lf.keys() & rf.keys()
    exact = wexact = wtotal = conflicts = crit_conf = crit_shared = 0.0
    for f in shared:
        inter = len(lf[f] & rf[f]); union = len(lf[f] | rf[f])
        sim = inter / union if union else 0.0
        w = FIELD_WEIGHT.get(f, 1.0); wtotal += w
        if sim == 1.0:
            exact += 1; wexact += w
        if sim == 0.0:
            conflicts += 1
            if f in crit: crit_conf += 1
        elif f in crit:
            crit_shared += 1
    nshared = ln.keys() & rn.keys()
    n_eq = n_conf = 0; rels = []
    for f in nshared:
        a, b = min(ln[f]), min(rn[f])
        rel = abs(a - b) / max(abs(a), abs(b), 1e-9)
        rels.append(rel)
        if rel <= 0.02: n_eq += 1
        else: n_conf += 1
    return [len(shared), exact / len(shared) if shared else 0.0,
            wexact / wtotal if wtotal else 0.0,
            conflicts, conflicts / len(shared) if shared else 0.0,
            crit_conf, float(crit_conf > 0), crit_shared,
            crit_shared / len(crit) if crit else 0.0,
            len(nshared), n_eq, n_conf,
            float(np.mean(rels)) if rels else -1.0,
            float(np.min(rels)) if rels else -1.0]

t = time.perf_counter()
CANON_X = np.array([canon_features(x, z, c) for x, z, c in zip(pairs.id1, pairs.id2, cat)],
                   dtype=np.float32)
log(f"признаки канонических полей: {CANON_X.shape} за {time.perf_counter()-t:.0f}с")

# Скоры новых моделей читаются на месте: качать выгрузку с 700 МБ весов нельзя.
bi_dir = os.path.dirname(glob.glob("/kaggle/input/**/ce_bi.npy", recursive=True)[0])
BI = np.load(f"{bi_dir}/ce_bi.npy").astype(np.float32)
hn_hits = glob.glob("/kaggle/input/**/ce_hardneg/llm_ood_scores.npy", recursive=True)
HN_FULL = np.load(hn_hits[0]).astype(np.float32)
src_pairs = pd.read_parquet(os.path.dirname(hn_hits[0]) + "/llm_ood_pairs.parquet")
pos_index = pd.Series(np.arange(len(src_pairs)),
                      index=pd.MultiIndex.from_arrays([src_pairs.id1, src_pairs.id2]))
take = pos_index.reindex(pd.MultiIndex.from_arrays([pairs.id1, pairs.id2])).to_numpy()
log(f"выравнивание скоров негативов: найдено {np.isfinite(take).mean():.1%}")
HN = HN_FULL[take.astype(int)]

CE = {n: np.load(f"{sc}/{n}.npy").astype(np.float32) for n in ("ce_relaxed", "ce_combo")}
codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)
masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos_, neg = rows[y[rows] == 1], rows[y[rows] == 0]
            keep = min(len(pos_), max(5, int(round(rate / (1 - rate) * len(neg)))))
            ch = np.concatenate([rng.choice(pos_, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

PARAMS = dict(max_iter=500, max_leaf_nodes=63, learning_rate=0.06, l2_regularization=1.0,
              early_stopping=False, random_state=0)
half = np.random.default_rng(5).permutation(len(y)) % 2
def run(extra, tag):
    mat = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], BI, HN, structural, codes]
                          + extra).astype(np.float32)
    p = np.zeros(len(y))
    for h in (0, 1):
        tr, te = half != h, half == h
        p[te] = HistGradientBoostingClassifier(**PARAMS).fit(mat[tr], y[tr]).predict_proba(mat[te])[:, 1]
    mu, sd = macro(rk(p))
    log(f"  {tag:<34} {mu:.6f} ± {sd:.6f}  ({mat.shape[1]} столбцов)")

run([], "как сейчас")
run([CANON_X], "плюс канонические поля")
